# Notebook 02. What does polydispersity do? Gold nanoparticles in DLS

**CoCoGel workshop tutorial, core part, about 30 minutes.** This notebook is self-contained. If you have just finished notebook 01, you can skim the how-to and primer sections, but do read the physics.

## Where we are

Notebook 01 ended with a question. BSA looks like one species, and a single exponential fits. **But real particles have a spread of sizes. Which radius does DLS report then?**

Here is the puzzle of this notebook. This dataset is for gold nanoparticles in water, with these numbers given:

> number-weighted **mean radius 30 nm**, relative standard deviation **15 %**.

We will do exactly what we did for BSA, and get a radius of about 34 nm. The particles have not changed. So **why does DLS say 34 nm when the mean is 30 nm?**

| Part | The question it answers | Minutes |
|---|---|---|
| Setup | How do I work in a notebook? | 3 |
| A | What radius does a single exponential give? | 7 |
| B | Is the shift visible by eye? | 3 |
| C | Who is DLS actually listening to? | 7 |
| D | Does the shape of the decay change as well? | 7 |
| E | Why does a 15 % spread look so small? | 3 |
| F | Try it yourself: what happens at larger spreads? | 3 |
| Bonus | For those with time | |

## What you will learn

- Why DLS reports an **intensity-weighted** size, and why that is larger than the number mean.
- How to compute a number and an intensity distribution in Python.
- How to look for a change of shape in a decay, with $\ln(g_2-1)$ and residuals.
- The **cumulant fit**, why its fit range must be limited, and what the PDI means.
- Why a PDI of 0.02 is not as "monodisperse" as it sounds.

## The physics, in words

Imagine many spheres of slightly different sizes, all diffusing in water. Small ones move fast and big ones move slowly. Each size contributes its own exponential decay. The detector does not see them with equal weight: **a big particle scatters far more light than a small one.** In the small-particle limit its intensity grows as $R^6$: double the radius, and you get 64 times the intensity.

So the correlation function is a sum of exponentials, weighted towards the big particles. Two things follow.

1. The decay is **slower** than for particles of the mean size, so the apparent radius is too large.
2. A sum of exponentials is **not** a single exponential, so the shape changes: fast at the start, slow at the end. This effect turns out to be small.

## The physics, in equations

**The observed length scale**, as in notebook 01:

$$q = \frac{4\pi n}{\lambda}\sin\frac{\theta}{2}, \qquad D(R) = \frac{k_BT}{6\pi\eta R}.$$

**The density correlation of a mixture.** Sizes $R_i$ contribute with weights $w_i$ that add up to 1:

$$f(q,\tau) = \sum_i w_i\,\exp\!\left(-D(R_i)\,q^2\tau\right), \qquad w_i \propto N_i\,I(R_i,q).$$

Here $N_i$ is the number of particles of size $R_i$, and $I(R_i,q)$ is the intensity scattered by one such particle. (Many books call $f$ the field correlation function $g_1$. We keep $f$.)

**What the detector records**, as in notebook 01:

$$g_2(q,\tau) = 1 + \beta\,|f(q,\tau)|^2 .$$

**The intensity-weighted averages.** With rates $\Gamma_i = D(R_i)q^2$, the mean rate and its variance are

$$\bar\Gamma = \sum_i w_i\Gamma_i, \qquad \mu_2 = \sum_i w_i\left(\Gamma_i-\bar\Gamma\right)^2, \qquad \mathrm{PDI} = \frac{\mu_2}{\bar\Gamma^2}.$$

**The apparent radius** is what we get by treating $\bar\Gamma$ as if there were one size:

$$R_\mathrm{app} = \frac{k_BT\,q^2}{6\pi\eta\,\bar\Gamma}.$$

Since $\Gamma\propto1/R$, this is a **harmonic** intensity-weighted mean, called the z-average: with $w_i\propto N_iR_i^6$,

$$R_\mathrm{app} = \frac{\sum N_iR_i^6}{\sum N_iR_i^5}.$$

## Every symbol and its unit

| Symbol | Meaning | Unit used here |
|---|---|---|
| $\theta$ | scattering angle | degree, converted to radian for `sin` |
| $n$ | refractive index of water | none (1.33) |
| $\lambda$ | wavelength in vacuum | nm |
| $q$ | scattering vector | nm⁻¹ |
| $\tau$ | lag time | s |
| $f(q,\tau)$ | normalised density correlation | none |
| $g_2(q,\tau)$ | normalised intensity correlation | none |
| $\beta$ | contrast, set by the optics | none |
| $\Gamma$ | decay rate of $f$ for a single-exponential fit | s⁻¹ |
| $\bar\Gamma$ | mean decay rate, weighted by intensity | s⁻¹ |
| $\mu_2$ | variance of the decay rates | s⁻² |
| PDI | polydispersity index $\mu_2/\bar\Gamma^2$ | none |
| $R$, $R_i$ | particle radius | nm |
| $N_i$ | number weight of size $R_i$ | none |
| $w_i$ | intensity weight of size $R_i$ | none |
| $D$ | diffusion coefficient | nm²/s, m²/s in Stokes–Einstein |
| $k_B$, $T$, $\eta$ | Boltzmann constant (J/K), temperature (K), viscosity (Pa·s) | |
| $R_\mathrm{app}$ | apparent radius, the z-average | nm |

## How a notebook works (JupyterLab and Colab)

A notebook is a page made of **cells**. There are two kinds:

- **Text cells** (Markdown), like the one you are reading. They explain things. Nothing is calculated in them.
- **Code cells**. They hold Python. On a laptop they have `[ ]:` on the left, and in Colab a round play button.

**To run a cell:** click on it and press **Shift+Enter** (hold Shift, press Enter). The output appears below it and the cursor moves to the next cell. On a Mac the same keys work.

**The rules that save you from most trouble:**

1. **Run the cells in order, from the top.** A cell can use only what earlier cells have already created. If you see `NameError: name '...' is not defined`, you skipped a cell.
2. **The number in `[3]:` tells you the order in which cells ran.** `[*]` means the cell is still running. Wait for it.
3. **Variables survive between cells.** Python remembers them until you restart. If you change a value in one cell, the cells below still hold the old results until you run them again.
4. **When things get strange, restart.**
   - JupyterLab: menu **Kernel**, then **Restart Kernel and Run All Cells...**
   - Colab: menu **Runtime**, then **Restart session and run all**.

   The *kernel* is the Python program behind the page. Restarting it empties its memory and costs nothing.
5. **You can't break anything.** The data files are never changed by this notebook. If you ruin a cell, download the notebook again.
6. **Save your work.** JupyterLab: Ctrl+S (Mac: Cmd+S). Colab: **File, Save a copy in Drive**, because a Colab session and its files are temporary.

**To reveal an answer,** click on a line that starts with a small triangle, such as "Answer" below a question. Try to answer first.

**You only ever need to edit lines marked `# <-- CHANGE ME`.** Everything else can stay as it is, unless you are curious, and then please do change it.

## A short Python primer: only what this notebook uses

Every command is explained again where it first appears, so you don't need to memorise anything.

**Variables.** A name for a value. `=` means "store".

```python
radius = 30.0              # a number. Everything after # is a comment
sample = "gold"            # a string (text)
```

**Lists.** Several values in order: `angles = [30, 90, 150]`.

**Indexing.** Python counts from **0**. `angles[0]` is `30`, and `angles[-1]` is the last element, `150`.

**Functions.** A named recipe. The indented lines belong to it, and `return` gives back the result.

```python
def square(x):
    return x * x
```

Powers use two stars: `x**2`. Numbers such as $1.38\times10^{-23}$ are written `1.38e-23`.

**`for` loops.** Repeat the indented lines for each element.

```python
for angle in [30, 90, 150]:
    print(angle)
```

**f-strings.** Text with values inside: `f"R = {radius:.1f} nm"` gives `R = 30.0 nm`. `:.1f` means one digit after the point, `:.3f` three, `:.0f` none, and `:.1%` means a percentage with one digit.

**Arrays.** A NumPy array is like a list on which arithmetic works element by element: if `R` is an array, `R**6` is an array holding each element to the sixth power. `np.sum(w * R)` multiplies element by element and then adds it all up, which is exactly $\sum_i w_iR_i$. This is why our formulas become so short.

**Indentation matters.** A missing or extra space at the start of a line gives `IndentationError`.

## Step 1. Find the data (this cell works on your laptop and in Colab)

**What this cell does and why.** The notebook needs the data files in the folder `data/`. On your laptop they are already there, because you downloaded the whole folder. On Google Colab the notebook arrives alone, without its data, so the cell has to fetch them. One notebook therefore serves both routes.

**Run it first, every time you open the notebook.** Click on the cell and press **Shift+Enter**.

Line by line:

| Code | Meaning |
|---|---|
| `import os, sys, subprocess` | `import` loads a library, which is a collection of ready-made tools. `os` talks to the operating system (folders and files). `sys` describes the running Python. `subprocess` starts other programs, here `git`. All three come with Python. |
| `URL = "..."`, `REPO = "..."` | Stores two pieces of text: the address of the materials on GitHub, and the name of the folder that will appear. Text in quotes is called a *string*. |
| `"google.colab" in sys.modules` | `sys.modules` is the list of libraries already loaded. Colab always loads one called `google.colab`, and a laptop does not. So this test is `True` only on Colab. |
| `os.path.exists("data")` | Asks: does a file or folder with this name exist here? The answer is `True` or `False`. |
| `if ... and not ...:` | The indented lines run only if both conditions hold: we are on Colab, and the data are not here yet. |
| `subprocess.run([...], check=True)` | Runs the program `git` with the words in the list. `git clone` copies a repository from the internet. `--depth 1` takes only the newest version, which is faster. `check=True` stops with a clear error if the copy fails. |
| `os.chdir(REPO)` | "Change directory": moves into the downloaded folder, so that `data/...` can be found. |
| `print(...)` | Writes text and values below the cell. |

In [ ]:
import os, sys, subprocess
URL = "https://github.com/harish-srini/cocogel_tutorial.git"   # where the materials live
REPO = "cocogel_tutorial"                            # the folder that git will create
if "google.colab" in sys.modules and not os.path.exists("data"):   # on Colab, data not here yet
    if not os.path.exists(REPO):                     # not downloaded yet
        subprocess.run(["git", "clone", "--depth", "1", URL], check=True)
    os.chdir(REPO)                                   # step into the downloaded folder
print("Data folder found:", os.path.exists("data"))

**What you should see.** One line: `Data folder found: True`. On Colab you may first see a short message from `git`.

**Common errors and fixes**

| Message or symptom | Cause | Fix |
|---|---|---|
| `Data folder found: False` on a laptop | JupyterLab was started in a different folder, so this notebook is not next to `data/`. | Close JupyterLab. In the terminal use `cd` to go to the folder holding this notebook, then type `jupyter lab` again. |
| `Data folder found: False` after unzipping | The notebook was opened from inside the ZIP, without extracting it. | Extract the ZIP first (Windows: right click, Extract All), and open the notebook from the extracted folder. |
| `CalledProcessError` on Colab | `git` could not reach GitHub. | Check your internet connection, then choose Runtime, Restart session, and run the cell again. |

## Step 2. Load the libraries

| Code | What it is | Why we need it here |
|---|---|---|
| `import numpy as np` | **NumPy**: calculations on whole arrays. `as np` is a short nickname. | exponentials, sums, the size grid |
| `import pandas as pd` | **pandas**: tables with named columns. | to read and select the data |
| `import matplotlib.pyplot as plt` | **matplotlib**: plots. | every figure |
| `from scipy.optimize import curve_fit` | **SciPy**: scientific tools. `from ... import ...` takes out one tool. | fitting |
| `plt.rcParams.update({...})` | Changes matplotlib's defaults. | readable fonts and sizes |

In [ ]:
import numpy as np                       # calculations with arrays of numbers
import pandas as pd                      # tables
import matplotlib.pyplot as plt          # plots
from scipy.optimize import curve_fit     # fitting
plt.rcParams.update({"font.size": 14, "figure.figsize": (7.5, 4.8)})   # readable figures
print("Libraries loaded")

**What you should see.** `Libraries loaded`.

**Common error.** `ModuleNotFoundError` means that the package is not installed in the Python JupyterLab is using. See the README, or switch to Colab.

## Step 3. All the analysis parameters, in one place

**What this cell does and why.** Everything that describes the sample and the optics is here, and nowhere else.

| Name | Meaning | Value and unit |
|---|---|---|
| `DATA_FILE` | the data file | text |
| `T` | temperature | 298.15 K (25 °C). K = °C + 273.15 |
| `WAVELENGTH_NM` | laser wavelength in vacuum | 633 nm |
| `N` | refractive index of water | 1.33, no unit |
| `ETA` | viscosity of water at 25 °C | 0.890 mPa·s = 0.890 × 10⁻³ Pa·s |
| `R_MEAN_NM` | number-weighted mean radius of this sample | 30 nm |
| `REL_SD` | relative standard deviation of the number distribution, as a fraction | 0.15, which is 15 % |
| `KB` | Boltzmann constant | 1.380649 × 10⁻²³ J/K. `np.pi` is the number π. |
| `KT_OVER_6PI_ETA` | the combination $k_BT/(6\pi\eta)$, converted so that $D$ in nm²/s equals this divided by $R$ in nm | nm³/s |

**The unit conversion in the last line.** $k_BT/(6\pi\eta)$ is in m³/s. One m³ is $10^{27}$ nm³, hence the factor `1e27`. Then $D = $ `KT_OVER_6PI_ETA / R` is in nm²/s when $R$ is in nm. We define this once and use it in every cell below, so all conversions live in this one place.

In [ ]:
# ===== ANALYSIS PARAMETERS: for another dataset, edit only this cell =====
DATA_FILE = "data/gold_nanoparticles_dls.csv"   # <-- CHANGE ME
T, ETA = 298.15, 0.890e-3                       # <-- CHANGE ME   K, Pa s (water, 25 degrees C)
WAVELENGTH_NM, N = 633.0, 1.33                  # <-- CHANGE ME   nm, refractive index
R_MEAN_NM, REL_SD = 30.0, 0.15                  # <-- CHANGE ME   number mean (nm), relative spread
KB = 1.380649e-23                               # J/K
KT_OVER_6PI_ETA = KB * T / (6 * np.pi * ETA) * 1e27    # nm^3/s, so D [nm^2/s] = this / R [nm]
print(f"D of a 30 nm sphere = {KT_OVER_6PI_ETA / 30 / 1e6:.2f} um^2/s")

**What you should see.** $D$ of a 30 nm sphere is about 8.18 μm²/s. That is about 9 times slower than BSA in notebook 01, as it must be: the radius is about 9 times larger.

**A new trick.** `T, ETA = 298.15, 0.890e-3` stores two values in one line: the first name gets the first value.

---
# Part A. A single exponential, as in notebook 01

## Step 4. Read and inspect the data

**What this cell does and why.** We look at the data before we fit.

- **`pd.read_csv(file)`** reads a CSV (comma-separated values) file into a table, called a **DataFrame**.
- **`df.head()`** gives the first five rows.
- **`df["angle_deg"].unique()`** gives each different value of a column once.

Because the last line of a cell is displayed automatically, the table appears without `print`.

| Column | Meaning | Unit |
|---|---|---|
| `angle_deg` | scattering angle $\theta$ | degree |
| `q_nm-1` | scattering vector $q$ | nm⁻¹ |
| `lag_time_s` | lag time $\tau$ | s |
| `g2` | intensity correlation $g_2$; tends to 1 at long lag | none |
| `g2_err` | standard error of `g2` | none |

In [ ]:
df = pd.read_csv(DATA_FILE)
print("Rows and columns:", df.shape)
print("Angles (degrees):", df["angle_deg"].unique())
df.head()

**What you should see.** 1224 rows and 5 columns, nine angles from 30° to 150°, and the first rows of the table. It has the same layout as the BSA file.

**Common errors.** `FileNotFoundError`: run the cell of Step 1 first. `NameError`: you skipped a cell, so run from the top.

## Step 5. Plot $g_2$ at a few angles

**What this cell does and why.** We look at the curves, as always.

- **`df[df["angle_deg"] == angle]`** keeps only the rows at that angle. The double `==` asks a question of each row, and `df[...]` keeps the rows where the answer is `True`.
- **`plt.semilogx(x, y, "o", ms=3, label=...)`** plots with a logarithmic x axis. `"o"` is a round marker and `ms` its size. `label` is the text for the legend.
- **`plt.xlabel`**, **`plt.ylabel`** give the axis labels, **`plt.legend`** the list of labels, and **`plt.show`** displays the figure. Text between `$` signs is typeset as maths, and the `r` in front of the string protects the backslashes.

**Unit conversion.** We plot $\tau$ in μs: seconds times $10^6$.

In [ ]:
chosen_angles = [30, 90, 150]                     # <-- CHANGE ME   any angles from the file
for angle in chosen_angles:
    d = df[df["angle_deg"] == angle]
    plt.semilogx(d["lag_time_s"] * 1e6, d["g2"], "o", ms=3, label=f"{angle}°")
plt.xlabel(r"lag time $\tau$ (μs)")
plt.ylabel(r"$g_2(q,\tau)$")
plt.legend(title=r"angle $\theta$")
plt.show()

**What you should see.** The same S-shape as for BSA, shifted about ten times to the right: at 90° the fall is near 200 μs and not 20 μs. Larger particles are slower. The curves look perfectly ordinary. Nothing in this plot warns you of a spread in size.

## Step 6. Fit each angle with a single exponential

**What this cell does and why.** We use the same model as in notebook 01, $g_2 = 1+\beta\exp(-2\Gamma\tau)$, with its factor of 2 because the detector records the square of $f$.

**Function.** `def g2_single(tau, beta, Gamma):` defines the model. `np.exp` is $e^x$, element by element. **For `curve_fit`, the first input must be the x variable,** and the others are the parameters to adjust.

**`curve_fit(model, x, y, p0=..., sigma=..., absolute_sigma=True, bounds=...)`:**

| Argument | Meaning |
|---|---|
| `model` | the function, with no round brackets |
| `x`, `y` | the data |
| `p0` | starting guess, in the order of the function's inputs. $\beta$: first point minus 1. $\Gamma$: $1/(2\tau_e)$, where $\tau_e$ is the lag at which $g_2-1$ has dropped by a factor $e$. |
| `sigma` | the error of each point |
| `absolute_sigma=True` | the errors are true sizes, not only relative weights |
| `bounds` | allowed ranges, `([lower], [upper])`. `np.inf` is infinity. |

**It gives back two things:** `popt`, the best parameters, and `pcov`, the covariance matrix, whose diagonal element `pcov[1, 1]` is the squared uncertainty of $\Gamma$. `np.sqrt` is the square root.

**Loop commands.** **`df.groupby("angle_deg")`** splits the table by angle, giving the angle and its part of the table in each round. **`.values`** gives a plain array. **`.iloc[0]`** takes the first element by position. **`np.argmin(np.abs(...))`** finds the position of the point closest to a target. **`rows.append({...})`** adds one row, and **`pd.DataFrame(rows)`** makes the table.

In [ ]:
def g2_single(tau, beta, Gamma):
    return 1 + beta * np.exp(-2 * Gamma * tau)
def fit_one_angle(d):
    tau, g2, err = d["lag_time_s"].values, d["g2"].values, d["g2_err"].values
    guess = [g2[0] - 1, 1 / (2 * tau[np.argmin(np.abs(g2 - 1 - (g2[0] - 1) / np.e))])]
    popt, pcov = curve_fit(g2_single, tau, g2, p0=guess, sigma=err,
                           absolute_sigma=True, bounds=([0, 0], [2, np.inf]))
    return popt[0], popt[1], np.sqrt(pcov[1, 1])       # beta, Gamma, uncertainty of Gamma

In [ ]:
rows = []
for angle, d in df.groupby("angle_deg"):
    beta, Gamma, Gamma_err = fit_one_angle(d)
    rows.append({"angle_deg": angle, "q": d["q_nm-1"].iloc[0], "beta": beta,
                 "Gamma": Gamma, "Gamma_err": Gamma_err})
fits = pd.DataFrame(rows)
fits

**What you should see.** A table with nine rows. $\Gamma$ rises from about 330 s⁻¹ at 30° to about 4730 s⁻¹ at 150°, and is about 2490 s⁻¹ at 90°.

**Common errors.** `RuntimeError: Optimal parameters not found`: the starting guess was poor, so check the data. `NameError`: you skipped a cell.

## Step 7. $\Gamma$ against $q^2$, and the apparent radius

**What this cell does and why.** As in notebook 01: if $\Gamma = Dq^2$, the slope gives $D$. For a line through zero with weights $w_i=1/\sigma_i^2$,

$$D = \frac{\sum_i w_iq_i^2\Gamma_i}{\sum_i w_iq_i^4}.$$

$\Gamma$ in s⁻¹ over $q^2$ in nm⁻² gives $D$ in nm²/s. Then the radius is `KT_OVER_6PI_ETA / D`, in nm, as prepared in Step 3. `plt.errorbar` plots points with error bars. `np.polyfit(x, y, 1, w=1 / y_err)` fits a straight line, with degree 1 and weights 1/error, and gives back the coefficients with the highest power first, so slope and intercept. We use it to check the intercept. `plt.plot([x1, x2], [y1, y2])` draws a line between two points.

In [ ]:
x, y, y_err = fits["q"].values**2, fits["Gamma"].values, fits["Gamma_err"].values
w = 1 / y_err**2
D_app = np.sum(w * x * y) / np.sum(w * x * x)                  # nm^2/s
R_app = KT_OVER_6PI_ETA / D_app                                # nm
print(f"D = {D_app / 1e6:.2f} um^2/s,  apparent radius = {R_app:.1f} nm,  mean = {R_MEAN_NM:.0f} nm")
print(f"intercept of a free line: {np.polyfit(x, y, 1, w=1 / y_err)[1]:.1f} per second")
plt.errorbar(x, y, yerr=y_err, fmt="o", label="fits")
plt.plot([0, x.max()], [0, D_app * x.max()], label=r"$\Gamma = Dq^2$")
plt.xlabel(r"$q^2$ (nm$^{-2}$)"); plt.ylabel(r"$\Gamma$ (s$^{-1}$)"); plt.legend(); plt.show()

**What you should see.** $D \approx 7.17\ \mathrm{\mu m^2/s}$ and an apparent radius of about 34 nm, with nine points on a straight line through the origin, and an intercept of about −6 s⁻¹, which is tiny next to rates of thousands per second.

**What it means.** The test of notebook 01 passes perfectly: this is diffusion. But the radius is **about 14 % above the mean radius of 30 nm**. The line gave no warning.

**Question 1.** Before reading further: is the error in the fit, in the viscosity, or in the particles? Is there anything wrong at all?

<details>
<summary>Answer</summary>

Nothing is wrong. The fit, the viscosity and the data are all fine. The 34 nm is the correct answer to a different question from the one we had in mind. We asked for the number-average radius, and DLS answers with an intensity-weighted one. The next parts show this.

</details>

**The open question:** a number can mislead. Is the difference between 30 and 34 nm actually visible in the data?

---
# Part B. Is the shift visible?

## Step 8. Overlay the curve for monodisperse 30 nm particles

**What this cell does and why.** If all particles had $R = 30$ nm exactly, the curve would follow directly from physics, with no fitting at all. $D = k_BT/(6\pi\eta R)$, then $\Gamma_{30} = Dq^2$. We draw it next to the data at one angle, with the same $\beta$ as the fit.

**`np.logspace(a, b, 300)`** gives 300 numbers evenly spaced on a logarithmic axis, from $10^a$ to $10^b$. **`np.log10`** is the base-10 logarithm. **`.iloc[0]`** on a selection of one row gives that row.

In [ ]:
ANGLE = 90                                                 # <-- CHANGE ME   any angle from the file
d = df[df["angle_deg"] == ANGLE]; fit = fits[fits["angle_deg"] == ANGLE].iloc[0]
tau, g2, err = d["lag_time_s"].values, d["g2"].values, d["g2_err"].values
Gamma_mono = KT_OVER_6PI_ETA / R_MEAN_NM * fit["q"]**2     # 1/s, for one size only
tau_fine = np.logspace(np.log10(tau.min()), np.log10(tau.max()), 300)
plt.semilogx(tau * 1e6, g2, "o", ms=3, label=f"data, {ANGLE}°")
plt.semilogx(tau_fine * 1e6, g2_single(tau_fine, fit["beta"], Gamma_mono), label="all at the mean R")
plt.xlabel(r"lag time $\tau$ (μs)"); plt.ylabel(r"$g_2$"); plt.legend(); plt.show()
print(f"Gamma: {Gamma_mono:.0f} at the mean radius,  {fit['Gamma']:.0f} from the fit")

**What you should see.** The model curve for 30 nm lies clearly to the **left** of the data: it decays too early. $\Gamma$ is about 2851 s⁻¹ for 30 nm and about 2492 s⁻¹ from the fit, which is a ratio of 1.14.

**What it means.** The shift is plain to see, and larger than the noise by far. **The sample decays as if the particles were bigger than they are.**

**The open question:** a spread of ±15 % has small and large particles in equal measure, roughly. So why does the decay follow the large ones?

---
# Part C. Who is DLS listening to?

## Step 9. Number distribution against intensity distribution

**What this cell does and why.** We calculate both distributions from the parameters of the sample.

**A lognormal distribution** is the bell curve in $\ln R$. It is a common description of particle sizes, because sizes can't be negative. Its two parameters follow from the mean and the relative standard deviation $\mathrm{RSD}$ of $R$:

$$s^2 = \ln\!\left(1+\mathrm{RSD}^2\right), \qquad \mu = \ln R_\mathrm{mean} - \frac{s^2}{2}.$$

**The size grid.** We take sizes spaced evenly in $\ln R$, from $\mu-8s$ to $\mu+8s$. Every bin has the same width in $\ln R$, so the number weight of each bin is simply proportional to the bell curve, and we need no bin-width factors. Eight widths is plenty, even after weighting with $R^6$.

**The weights.** `w_number` is the bell curve, normalised to add up to 1. `w_intensity` is `w_number * R**6`, normalised. That $R^6$ is the small-particle limit: the scattered amplitude of a small sphere goes as its volume, $R^3$, and the intensity as the square of the amplitude.

**Commands.** **`np.log`** is the natural logarithm. **`np.linspace(a, b, n)`** gives `n` evenly spaced numbers from `a` to `b`. **`np.exp`** is its inverse, so `R = np.exp(x)` gives the radii. `return R, w_number, w_intensity` gives back three things at once.

**Why a function?** We will need the same calculation again for other spreads in Part F.

In [ ]:
def size_distribution(polydispersity, mean=R_MEAN_NM):
    s = np.sqrt(np.log(1 + max(polydispersity, 1e-3)**2))     # width in ln R; max() avoids s = 0
    mu = np.log(mean) - s**2 / 2                              # centre in ln R
    x = np.linspace(mu - 8 * s, mu + 8 * s, 2001)             # evenly spaced in ln R
    w_number = np.exp(-(x - mu)**2 / (2 * s**2))              # the bell curve
    w_intensity = w_number * np.exp(x)**6                     # number times R^6
    return np.exp(x), w_number / w_number.sum(), w_intensity / w_intensity.sum()
R, w_number, w_intensity = size_distribution(REL_SD)

**What you should see.** Nothing is printed: the cell only defines things. No output is fine.

`max(a, b)` gives the larger of two values. It protects the formula at zero spread, since `s = 0` would mean dividing by zero. A spread of 0.001 is as good as zero here. `.sum()` adds all elements of an array.

## Step 10. Plot the two distributions, and compute the averages

**What this cell does and why.** We compare the distributions by eye and by number.

The averages are direct translations of the formulas:

- number mean: $\sum N_iR_i$, in code `np.sum(w_number * R)`. Check: it must return 30 nm.
- z-average: $\sum N_iR_i^6/\sum N_iR_i^5$. With intensity weights this is $1/\sum_i w_i(1/R_i)$, in code `1 / np.sum(w_intensity / R)`.

**`plt.plot(x, y)`** draws a line through the points. We plot each curve divided by its own maximum, `.max()`, so that both reach 1 and can be compared. The vertical lines are drawn with **`plt.axvline(x)`**, and **`plt.xlim(a, b)`** sets the range of the x axis.

In [ ]:
R_number = np.sum(w_number * R)                     # number mean, nm
R_z = 1 / np.sum(w_intensity / R)                   # z-average with R^6 weights, nm
print(f"number mean = {R_number:.1f} nm, z-average (R^6) = {R_z:.1f} nm, from the fit = {R_app:.1f} nm")
plt.plot(R, w_number / w_number.max(), label="number")
plt.plot(R, w_intensity / w_intensity.max(), label=r"intensity ($\propto R^6$)")
plt.axvline(R_number, color="C0", ls="--"); plt.axvline(R_z, color="C1", ls="--")
plt.xlim(10, 70); plt.xlabel("radius $R$ (nm)"); plt.ylabel("weight (peak = 1)"); plt.legend(); plt.show()
big = R > 40                                        # True for radii above 40 nm
print(f"Above 40 nm: {w_number[big].sum():.1%} of number, {w_intensity[big].sum():.1%} of intensity")

**What you should see.** Two bell-like curves. The intensity curve is shifted to the right by several nm. Number mean 30.0 nm, z-average with $R^6$ weights 33.5 nm, and the fit about 34 nm. Particles above 40 nm are about 2.3 % of the number and about 13 % of the intensity.

**How to read it: this answers the puzzle.** The particles that dominate the signal are not the typical ones. **A few large particles are heard loudest.** A particle of 40 nm is $(40/30)^6 = 5.6$ times brighter than one of 30 nm. DLS answers truthfully, but its question is "what is the mean rate, weighted by intensity?" and not "what is the mean size?".

`w_number[big]` is another use of a `True`/`False` filter: it keeps only the elements where `big` is `True`. `ls="--"` makes the line dashed. `"C0"` and `"C1"` are matplotlib's first two default colours.

**Note: $R^6$ against the full Mie result.** $R^6$ holds only for spheres much smaller than the wavelength. A 30 nm gold sphere is close to, but not fully in, that limit, and gold's optical response is near a resonance at 633 nm. A full calculation, Mie theory, solves Maxwell's equations for a sphere, using gold's complex refractive index, $0.18 + 3.43i$ at 633 nm (P. B. Johnson, R. W. Christy, *Phys. Rev. B* **6**, 4370, 1972). The local power of the Mie intensity, $d\ln I/d\ln R$ at 90°, is:

| $R$ (nm) | 10 | 20 | 30 | 40 | 50 | 60 | 70 | 100 |
|---|---|---|---|---|---|---|---|---|
| power | 6.1 | 6.3 | 6.7 | 6.9 | 6.1 | 3.7 | 1.6 | 0.1 |

So near 30–50 nm the intensity rises slightly **faster** than $R^6$, and beyond about 60 nm it levels off. For this sample the full result is $R_\mathrm{app} = 34.1$ nm and PDI = 0.0227, against 33.5 nm and 0.0225 from $R^6$. That is the small difference between the z-average in the cell above and the 34 nm of the fit. $R^6$ is a good teaching tool here, but only for narrow distributions around this size, as Part F will show.

**Common errors.** `NameError: name 'R' is not defined`: you skipped the previous cell. An odd-looking plot: check the `xlim` range.

**Question 2.** Using $R^6$, what is the ratio of the intensities of a 60 nm particle and a 30 nm particle? How many 30 nm particles equal one 60 nm particle in signal?

<details>
<summary>Answer</summary>

2⁶ = 64. So one 60 nm particle scatters like 64 of the 30 nm particles. This is also why a trace of dust or aggregates can ruin a DLS measurement of small proteins, as Question 8 in notebook 01 hinted.

</details>

**Question 3.** From the formula for $R_\mathrm{app}$: is the z-average affected more by the small or by the large side of the distribution?

<details>
<summary>Answer</summary>

By the large side, strongly: both the numerator (R⁶) and the denominator (R⁵) are dominated by large particles. Removing all particles below 25 nm would hardly change it, while adding a few above 45 nm would raise it noticeably.

</details>

**The open question:** a size spread shifts the decay. A sum of exponentials isn't an exponential, so can we also see the spread in the **shape** of the curve? That would be useful, since then DLS could reveal the width on its own.

---
# Part D. Looking for the change of shape

## The expected effect

Take the logarithm of $f$ and expand in powers of $\tau$. This is the **cumulant expansion**:

$$\ln f(q,\tau) = -\bar\Gamma\tau + \frac{\mu_2}{2}\tau^2 - \dots$$

- The first term is a straight line with slope $-\bar\Gamma$: the mean rate.
- The second term, with the variance $\mu_2$ of the rates, bends the line upwards. The fast decays die first, and the slow ones are left behind.

Since $g_2-1 = \beta f^2$, doubling everything,

$$\ln\!\left(g_2-1\right) = \ln\beta - 2\bar\Gamma\tau + \mu_2\tau^2 .$$

A pure exponential gives a straight line in $\ln(g_2-1)$ against $\tau$, on a **linear** $\tau$ axis. Polydispersity gives a slight upward curvature.

## Step 11. $\ln(g_2-1)$ against $\tau$, with residuals

**What this cell does and why.** We plot the data in the form where a single exponential is straight, and show the residuals from the single-exponential fit.

**Avoiding the logarithm of noise.** Once $g_2-1$ has reached the noise it can be negative, and its logarithm is undefined. So we plot only where $g_2-1$ is well above zero, `mask = g2 - 1 > 0.02`, and only up to that point.

**Commands.** **`np.log`** is the natural logarithm. `tau[mask]` keeps only the elements where `mask` is `True`. **`plt.subplots(2, 1, sharex=True)`** makes two panels, one above the other, sharing an x axis. **`.errorbar`** draws error bars. **`.axhline(0)`** a horizontal line at 0. `height_ratios` sets the relative heights. With panels, the labels use `set_xlabel` and `set_ylabel`.

**Unit.** We plot $\tau$ in μs. `tau * 1e6` converts s to μs.

In [ ]:
mask = g2 - 1 > 0.02                                 # keep only where the signal is well above noise
t_us, residual = tau[mask] * 1e6, (g2 - g2_single(tau, fit["beta"], fit["Gamma"]))[mask]
fig, (top, bottom) = plt.subplots(2, 1, sharex=True, figsize=(7.5, 6), height_ratios=[2, 1])
top.plot(t_us, np.log(g2[mask] - 1), "o", ms=3, label=f"data, {ANGLE}°")
top.plot(t_us, np.log(fit["beta"]) - 2 * fit["Gamma"] * tau[mask], label="single exponential")
bottom.errorbar(t_us, residual, yerr=err[mask], fmt="o", ms=3); bottom.axhline(0, color="k")
top.legend(); top.set_ylabel(r"$\ln(g_2-1)$"); bottom.set_ylabel(r"$g_2$ − fit")
bottom.set_xlabel(r"lag time $\tau$ (μs)"); plt.show()

**What you should see.** In the top panel, a line that looks straight, with the data on it. You can't see any curvature. In the bottom panel, the residuals are only one or two parts in a thousand, but they are not random: they dip below zero near 100 μs and rise above zero near 500 μs, by about 0.0015.

**How to read it: at a spread of 15 %, the change of shape is subtle.** The top panel alone could not tell this sample from a monodisperse one. The wave in the residuals is the true sign of the spread, and it is about one to two error bars high. The larger effect is the shift in apparent size that you saw in Part B.

**The open question:** a wave in the residuals is only a hint. Can a fit turn it into a number?

## Step 12. The cumulant fit, in a limited range

**What this cell does and why.** We fit the expansion directly:

$$g_2 = 1 + \beta\exp\!\left(-2\bar\Gamma\tau + \mu_2\tau^2\right).$$

**Why the fit range must be limited.** The cumulant series is an expansion for small $\bar\Gamma\tau$. At large $\tau$ the term $\mu_2\tau^2$ grows without limit, and the formula would eventually rise, which no correlation function does. So the formula may only be used where the next terms are still small.

We use $\tau < 1.5/\Gamma$, where $f$ has dropped to $e^{-1.5} \approx 0.22$, and $g_2-1$ to about 5 % of $\beta$. Beyond that the signal also sinks into the noise.

- **Too short a range:** a short piece of a curve hardly shows curvature, so $\mu_2$ is very noisy.
- **Too long a range:** the missing higher terms bias $\mu_2$ low, because the true curve is not a parabola.

**Commands.** `FIT_RANGE` holds the limit, `keep = tau < FIT_RANGE / fit["Gamma"]` makes the `True`/`False` selection, and `tau[keep]` applies it. `popt[2]` is $\mu_2$. Its unit is s⁻², so PDI = $\mu_2/\bar\Gamma^2$ has no unit.

**The returned values.** As before, `popt` holds the best parameters in the order of the function's inputs, and the square roots of the diagonal of `pcov` hold their uncertainties. `np.diag` picks out the diagonal.

In [ ]:
FIT_RANGE = 1.5                                        # <-- CHANGE ME   fit up to Gamma tau = this
def g2_cumulant(tau, beta, Gamma_bar, mu2):
    return 1 + beta * np.exp(-2 * Gamma_bar * tau + mu2 * tau**2)
keep = tau < FIT_RANGE / fit["Gamma"]
popt, pcov = curve_fit(g2_cumulant, tau[keep], g2[keep], p0=[fit["beta"], fit["Gamma"], 0.0],
                       sigma=err[keep], absolute_sigma=True)
pdi, pdi_err = popt[2] / popt[1]**2, np.sqrt(pcov[2, 2]) / popt[1]**2
print(f"{keep.sum()} points. mean rate = {popt[1]:.0f} per second, PDI = {pdi:.3f} +- {pdi_err:.3f}")
print(f"apparent radius from the mean rate = {KT_OVER_6PI_ETA * fit['q']**2 / popt[1]:.1f} nm")

**What you should see, at 90°.** 75 points, a mean rate of about 2513 s⁻¹, PDI of about $0.026 \pm 0.005$, and an apparent radius of about 34.0 nm.

**How to read it.** The PDI is non-zero by several times the printed uncertainty. But as you learned in notebook 01, the simple error bars from `curve_fit` are too small, because neighbouring lags share noise. To see how much to trust this number, compare angles.

**Common error.** `ValueError: `ydata` must not be empty`: you made `FIT_RANGE` too small, so no points were left.

## Step 13. The cumulant fit at all nine angles

**What this cell does and why.** The physics says that PDI should barely depend on angle, so nine angles give nine estimates of one number. Their scatter is an honest measure of the uncertainty, one that doesn't rely on any error bar.

The standard error of a mean of $n$ values is the standard deviation divided by $\sqrt{n}$. **`.std()`** gives the standard deviation, and **`.mean()`** the mean. **`.std(ddof=1)`** uses $n-1$ in the formula, the usual choice for a sample.

In [ ]:
def fit_cumulant(d, f0):
    t, g, e = d["lag_time_s"].values, d["g2"].values, d["g2_err"].values
    k = t < FIT_RANGE / f0["Gamma"]
    p, c = curve_fit(g2_cumulant, t[k], g[k], p0=[f0["beta"], f0["Gamma"], 0.0],
                     sigma=e[k], absolute_sigma=True)
    return p[1], p[2] / p[1]**2, np.sqrt(c[2, 2]) / p[1]**2     # mean rate, PDI, its uncertainty

In [ ]:
rows = []
for angle, d in df.groupby("angle_deg"):
    f0 = fits[fits["angle_deg"] == angle].iloc[0]
    G_bar, pdi_a, pdi_a_err = fit_cumulant(d, f0)
    rows.append({"angle_deg": angle, "Gamma_bar": G_bar, "PDI": pdi_a, "PDI_err": pdi_a_err,
                 "R_app_nm": KT_OVER_6PI_ETA * f0["q"]**2 / G_bar})
cum = pd.DataFrame(rows)

In [ ]:
se = cum["PDI"].std(ddof=1) / np.sqrt(len(cum))        # standard error of the mean
print(f"mean PDI = {cum['PDI'].mean():.3f} +- {se:.3f}  (nine angles)")
cum

**What you should see.** A table of nine rows. The PDI ranges from about −0.005 at 30° to about 0.033 at 45°. The mean is about $0.020 \pm 0.004$. $R_\mathrm{app}$ stays near 34 nm at all angles, between about 33.3 nm and 34.7 nm.

**How to read it**

- **A single angle can mislead.** The values at neighbouring angles differ by much more than their printed errors. The value at 30° is even slightly negative, which is impossible for a variance. It simply means that the noise there is as large as the signal. **Never report a PDI from one fit without asking how reproducible it is.**
- **Together they speak clearly.** The mean is about five standard errors above zero, and it agrees with the expected 0.0225–0.023.
- **The angle dependence is very weak.** The full Mie calculation for this sample predicts $R_\mathrm{app}$ from 34.07 to 34.08 nm and PDI from 0.0227 to 0.0228 between 30° and 150°: no signature you could ever detect. At this size, the particles are too small for their form factor to change with angle. **So angle dependence is no test for polydispersity here.** It would become one for much larger particles.

Also note: the $R_\mathrm{app}$ from the cumulant fit at 90° is a bit below the value from the single exponential. A single-exponential fit over the full range leans towards the slow tail, so it reads slightly larger.

**Question 4.** Change `FIT_RANGE` to 0.75 and then to 3, and re-run Steps 12 and 13. What happens to the mean PDI and its standard error? Set it back to 1.5 afterwards.

<details>
<summary>Answer</summary>

With 0.75 the fit sees little curvature: fewer points, and the PDI values scatter much more between angles, so the standard error rises to about 0.01. With 3 the standard error stays about the same as at 1.5, but the mean drops to about 0.016, because the second-order formula is used outside its range and the missing higher terms bias it low. The choice of range is a trade-off between noise and bias. A good report states the range.

</details>

**The open question:** we found PDI ≈ 0.02, for a particle population with a spread of ±15 %. Those two numbers don't look alike. Is one of them wrong?

---
# Part E. The surprise: PDI is the *square* of the width

Take the $R^6$ limit, with $\Gamma\propto 1/R$, and a lognormal distribution with RSD. Using $\langle R^m\rangle = \exp(m\mu + m^2s^2/2)$ for the number average, a few lines of algebra give an exact result:

$$\mathrm{PDI} = \frac{\mu_2}{\bar\Gamma^2} = \frac{\langle R^4\rangle\langle R^6\rangle}{\langle R^5\rangle^2}-1 = e^{s^2}-1 = \mathrm{RSD}^2 .$$

The PDI is the **square** of the relative width. Squaring a small number makes it much smaller.

## Step 14. The table every DLS user should see

**What this cell does and why.** We compute the PDI numerically from `size_distribution`, and compare with RSD².

The PDI from weights: rates are $\propto1/R$, so $\mu_2/\bar\Gamma^2$ is $\langle1/R^2\rangle_w/\langle1/R\rangle_w^2 - 1$, which the function below translates directly. It uses the weights from Step 9.

In [ ]:
def pdi_and_radius(polydispersity):
    R, w_n, w_i = size_distribution(polydispersity)
    mean_inv = np.sum(w_i / R)                       # intensity-weighted mean of 1/R
    return np.sum(w_i / R**2) / mean_inv**2 - 1, 1 / mean_inv     # PDI, apparent radius in nm
for rsd in [0.05, 0.10, 0.15, 0.20, 0.30]:
    pdi_value, r_value = pdi_and_radius(rsd)
    print(f"spread {rsd:4.0%}: PDI = {pdi_value:.4f}, spread^2 = {rsd**2:.4f}, radius = {r_value:.1f} nm")

**What you should see.** The PDI equals the square of the spread in every row: 0.0025, 0.0100, 0.0225, 0.0400, 0.0900.

**How to read it.** Our sample, with a spread of **15 %**, has a PDI of only **0.02**. Values below 0.05 are usually taken to mean a highly monodisperse sample; one instrument guide says that they are "rarely seen apart from for highly monodisperse standards" (Malvern Panalytical, *Dynamic light scattering: common terms defined*). By that standard, a sample with sizes spread by ±15 %, which on an electron micrograph would be easy to see, could pass as almost perfectly uniform.

**The lesson.** A small PDI doesn't mean a narrow distribution in the sense you might expect. Take the square root of the PDI for a rough relative width. $\sqrt{0.02} \approx 14\,\%$.

`:4.0%` formats a fraction as a percentage with no digits after the point, in a space of 4 characters, so that the columns line up.

**Question 5.** A report states PDI = 0.1. What relative width of the distribution does this suggest? And what would you expect its $R_\mathrm{app}$ to be, relative to the number mean?

<details>
<summary>Answer</summary>

√0.1 ≈ 32 % relative width. For a lognormal distribution in the R⁶ limit, R_app / R_mean = exp(5s²) = (1 + RSD²)⁵ = 1.1⁵ ≈ 1.61. So the apparent radius would be about 60 % above the number mean, though the R⁶ assumption gets weaker for such widths.

</details>

**The open question:** how do shift and shape change together as the spread grows? Now try it yourself.

---
# Part F. Try it yourself

## Step 15. Change the spread

**What this cell does and why.** The model gets the spread from the line marked `# <-- CHANGE ME`. It calculates $f(\tau) = \sum_iw_ie^{-D_iq^2\tau}$ and draws $g_2$, along with the data and the curve for monodisperse 30 nm.

**How the sum is written.** `rates` holds $\Gamma_i=D_iq^2$ for all sizes. **`np.outer(tau, rates)`** makes a table with one row per lag and one column per size, holding $\tau\Gamma_i$. `np.exp(-...)` applies the exponential to the whole table. **`@ w`** then does the sum over sizes with the weights, row by row: this `@` is matrix multiplication, and here it is exactly $\sum_i w_ie^{-\Gamma_i\tau}$.

**Try:** `0`, `0.15`, `0.3` and `0.5`. After each change, press Shift+Enter on this cell, and look at the plot and the printed numbers.

In [ ]:
polydispersity = 0.15  # <-- CHANGE ME
R_m, w_n, w_i = size_distribution(polydispersity)
rates = KT_OVER_6PI_ETA / R_m * fit["q"]**2                 # Gamma_i, in 1/s
g2_mix = 1 + fit["beta"] * (np.exp(-np.outer(tau_fine, rates)) @ w_i)**2
pdi_model, r_model = pdi_and_radius(polydispersity)
print(f"spread = {polydispersity:.0%}:  apparent radius = {r_model:.1f} nm,  PDI = {pdi_model:.4f}")
plt.semilogx(tau * 1e6, g2, "o", ms=3, color="0.6", label="data")
plt.semilogx(tau_fine * 1e6, g2_mix, label=f"model, spread {polydispersity:.0%}")
plt.semilogx(tau_fine * 1e6, g2_single(tau_fine, fit["beta"], Gamma_mono), "--", label="all at mean R")
plt.xlabel(r"lag time $\tau$ (μs)"); plt.ylabel(r"$g_2$"); plt.legend(); plt.show()

**What you should see**

| `polydispersity` | $R_\mathrm{app}$ from $R^6$ (this cell) | PDI from $R^6$ (this cell) | $R_\mathrm{app}$ from full Mie, 90° | PDI from full Mie, 90° |
|---|---|---|---|---|
| 0 | 30.0 nm | 0.0000 | 30.0 nm | 0.0000 |
| 0.15 | 33.5 nm | 0.0225 | 34.1 nm | 0.0227 |
| 0.30 | 46.2 nm | 0.0900 | 45.3 nm | 0.0656 |
| 0.50 | 91.6 nm | 0.2500 | 57.6 nm | 0.0907 |

(The Mie columns come from a separate full calculation and are given here for comparison. They don't come from this cell. The printed "spread = 0%" actually uses the tiny value 0.001 inside the function.)

**How to read it**

- At **0** the model falls on the dashed curve, of course, to the left of the data.
- At **0.15** the model lies close to the data. It sits slightly to the left of them, because $R^6$ gives 33.5 nm and not the 34 nm of the full calculation.
- At **0.3** the model has moved well to the right, and the shape has visibly stretched.
- At **0.5** the curve is shifted strongly, and its long tail is obvious.

**Be careful at 0.5.** Compare the two columns: $R^6$ says 91.6 nm and full Mie says 57.6 nm. At large spreads the big particles are outside the $R^6$ limit, since beyond about 60 nm their intensity levels off. So $R^6$ overstates both their influence and the shift. The honest statement: $R^6$ is a sound guide up to about 15–20 % spread at this size, and is only qualitative beyond that.

**Common errors.** `NameError: name 'tau_fine' is not defined`: you skipped Step 8. A negative value in `polydispersity` makes no physical sense.

**Note.** Real particles also carry a surface layer (ligands, stabilisers and bound water) that adds to their friction, and so to $R_h$, though it hardly scatters at all compared with gold. Part of a real gap between a radius from electron microscopy and one from DLS is due to this layer, on top of the intensity weighting. The present calculation has only the size spread.

---
# What you have done

Read the chain again:

1. **Part A:** a single exponential passes every test of notebook 01, and returns about 34 nm for particles with a mean of 30 nm. *Why?*
2. **Part B:** the shift is plainly visible against a monodisperse 30 nm curve. *Why does the decay follow the large particles?*
3. **Part C:** intensity weighting: a few large particles dominate. *Is the spread also visible in the shape?*
4. **Part D:** barely. The cumulant fit needs care, and only the average over angles gives a clear PDI, about 0.02. *Why so small, for a spread of 15 %?*
5. **Part E:** because PDI ≈ (relative width)².
6. **Part F:** shift and shape grow quickly with the width, and $R^6$ eventually gives way to full Mie scattering.

**The message to carry to your own samples.**

- DLS reports an intensity-weighted size. It is not wrong, but it answers a particular question.
- At a moderate spread, the **shift in apparent size** is the larger effect, and the change in shape is subtle.
- A small PDI can hide a real width. Take its square root before calling a sample monodisperse.
- Angle dependence doesn't help for particles of this size.

**The question that leads on.** All of this assumes particles that ignore each other. What happens when they don't? **That is the optional notebook 03.**

---
# Bonus exercises

*For those with time or Python experience.*

**Exercise 1.** Find the size above which particles carry half of the intensity, and the share of the number they represent. Use `np.cumsum`, which gives a running sum, and `np.argmax` on a `True`/`False` array, which gives the position of the first `True`.

<details>
<summary>Answer</summary>

```python
R_e, wn_e, wi_e = size_distribution(0.15)
i_half = np.argmax(np.cumsum(wi_e) >= 0.5)
print(R_e[i_half], wn_e[i_half:].sum())
```

The median of the intensity lies near 33.9 nm in the R⁶ limit, and only about 19 % of the particles by number are larger than this. Half of the signal comes from less than one fifth of the particles.

</details>

**Exercise 2.** How precisely can you recover the PDI from noisy data? Add noise to a model $g_2$ with `np.random.default_rng(1).normal(0, sigma, size)`, fit it as in Step 12, and repeat 100 times. How does the spread of the PDI depend on the noise level `sigma`?

<details>
<summary>Answer</summary>

This fits a noise-free curve with independent noise added. Real correlators have correlated noise, so the true spread is larger than what you find here.

```python
rng = np.random.default_rng(1)
R_x, wn_x, wi_x = size_distribution(0.15)
Gi_x = KT_OVER_6PI_ETA / R_x * fit["q"]**2
t_x = tau[tau < 1.5 / fit["Gamma"]]
clean = 1 + 0.8 * (np.exp(-np.outer(t_x, Gi_x)) @ wi_x)**2
for sigma in [0.0005, 0.002]:
    out = []
    for i in range(100):
        p_x, _ = curve_fit(g2_cumulant, t_x, clean + rng.normal(0, sigma, t_x.size), p0=[0.8, Gi_x.mean(), 0.0])
        out.append(p_x[2] / p_x[1]**2)
    print(sigma, np.mean(out), np.std(out))
```

The spread of the PDI grows in proportion to the noise. You should find about 0.003 at σ = 0.0005, and about 0.01 at σ = 0.002. So with a PDI of 0.02 you need noise well below 0.002 in g₂ for one curve to be meaningful.

</details>

**Exercise 3.** A sample has 99.9 % by number of 30 nm particles and 0.1 % of 300 nm aggregates. Using $R^6$, what share of the intensity comes from the aggregates? (Remember: this far from the small-particle limit, the true ratio is smaller, but the conclusion stands.)

<details>
<summary>Answer</summary>

Intensity ratio per particle: 10⁶. So (0.001 × 10⁶) / (0.001 × 10⁶ + 0.999) ≈ 99.9 % of the intensity comes from 0.1 % of the particles. DLS would mostly report the aggregates. This is why samples for DLS are filtered.

</details>

**Exercise 4.** Fit two exponentials to the data at 90°, with $f = A\,e^{-\Gamma_1\tau} + (1-A)\,e^{-\Gamma_2\tau}$. Do you get two clearly different rates? Why should you be suspicious of them, even if the fit looks good?

<details>
<summary>Answer</summary>

```python
def g2_two(t, beta, A, G1, G2):
    return 1 + beta * (A * np.exp(-G1 * t) + (1 - A) * np.exp(-G2 * t))**2
p2, _ = curve_fit(g2_two, tau, g2, p0=[fit["beta"], 0.5, 1.4 * fit["Gamma"], 0.7 * fit["Gamma"]], sigma=err,
                  bounds=([0, 0, 0, 0], [2, 1, np.inf, np.inf]))
print(p2)
```

Here you should get a main rate of about 2460 s⁻¹, and about 1 % of a fast one near 7600 s⁻¹. The fit returns two rates even though the sample has one broad distribution and no two populations. Two peaks from a fit are not proof of two populations. Fitting a sum of exponentials to a smooth decay is a notoriously ill-conditioned problem: very different sets of exponentials give nearly the same curve within the noise.

</details>